In [18]:
import json
import numpy as np
import pandas as pd

METRICS_PATH = "metrics.jsonl"

# --- dataset sizes (bytes) ---
subset_bytes_1 = 56_298_639_360
subset_bytes_2 = 230_371_246_080
total_bytes  = 3_722_450_821_120

# --- if you already computed raw total tokens (C * seconds) ---
total_raw_tokens = 9_306_127_052  # <- 너가 이전에 계산한 값

# --- world size (DDP GPUs) ---
world_size = 2  # calibration은 1로 두고, DDP면 2로 바꾸면 됨

rows = []
with open(METRICS_PATH, "r") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)

# 1) bucket distribution
print("bucket_P counts:", df["bucket_P"].value_counts().to_dict())
print("bucket_C counts:", df["bucket_C"].value_counts().to_dict())

# 2) basic stats
tokens_per_update = int(df["tokens_per_update"].iloc[-1])
step_time_mean = float(df["steps_time"].mean())

tgt_frac = (df["tgt_tokens_sum"] / (df["tgt_tokens_sum"] + df["ctx_tokens_sum"])).mean()
print(f"tokens_per_update={tokens_per_update} (basis={df['accum_basis'].iloc[-1]})")
print(f"mean step_time={step_time_mean:.2f}s")
print(f"mean target fraction (tgt/(ctx+tgt))={tgt_frac:.4f}")

# 3) throughput
target_tps = tokens_per_update / step_time_mean
total_tps  = (tokens_per_update / tgt_frac) / step_time_mean
print(f"throughput: target_toks/s={target_tps:,.0f}, total_toks/s≈{total_tps:,.0f}")

print()
# 5) steps per epoch (raw tokens -> target tokens -> steps)
total_tgt_tokens = total_raw_tokens * tgt_frac
steps_per_epoch = total_tgt_tokens / (tokens_per_update * world_size)
print(f"FULL: target_tokens≈{total_tgt_tokens/1e9:.3f}B, steps/epoch≈{steps_per_epoch:,.0f} (world_size={world_size})")

for subset_bytes in [subset_bytes_1, subset_bytes_2]:
    # 4) subset fraction by bytes
    f = subset_bytes / total_bytes
    print(f"subset fraction by bytes: {f*100:.3f}%")

    subset_raw_tokens = total_raw_tokens * f
    subset_tgt_tokens = subset_raw_tokens * tgt_frac
    subset_steps_epoch = subset_tgt_tokens / (tokens_per_update * world_size)
    print(f"SUBSET: raw_tokens≈{subset_raw_tokens/1e6:.1f}M, target_tokens≈{subset_tgt_tokens/1e6:.1f}M, steps/epoch≈{subset_steps_epoch:,.0f}")

# 6) if you want “10% raw token budget” step estimate
# frac = 0.10
# steps_10pct = (total_raw_tokens * frac * tgt_frac) / (tokens_per_update * world_size)
# print(f"10% raw token budget -> steps≈{steps_10pct:,.0f}")

bucket_P counts: {30: 121, 10: 79}
bucket_C counts: {128: 168, 60: 20, 19: 5, 213: 3, 198: 2, 129: 2}
tokens_per_update=131072 (basis=target)
mean step_time=8.87s
mean target fraction (tgt/(ctx+tgt))=0.3396
throughput: target_toks/s=14,777, total_toks/s≈43,515

FULL: target_tokens≈3.160B, steps/epoch≈12,055 (world_size=2)
subset fraction by bytes: 1.512%
SUBSET: raw_tokens≈140.7M, target_tokens≈47.8M, steps/epoch≈182
subset fraction by bytes: 6.189%
SUBSET: raw_tokens≈575.9M, target_tokens≈195.6M, steps/epoch≈746


In [26]:
import numpy as np
import pandas as pd

# Load
df = pd.read_json("metrics.jsonl", lines=True)

# ---------- Basic derived columns ----------
df["valid_tokens_step"] = df["ctx_tokens_sum"].astype(np.int64) + df["tgt_tokens_sum"].astype(np.int64)
df["tgt_ratio"] = df["tgt_tokens_sum"] / df["valid_tokens_step"].clip(lower=1)

# Throughput (tokens/sec, samples/sec)
df["tokens_per_sec"] = df["valid_tokens_step"] / df["steps_time"].clip(lower=1e-9)
df["samples_per_sec"] = df["batch_size_samples"] / df["steps_time"].clip(lower=1e-9)

# ---------- Summary stats helper ----------
def qstats(s: pd.Series, qs=(0.1, 0.5, 0.9, 0.95, 0.99)):
    s = pd.to_numeric(s, errors="coerce").dropna()
    out = {"count": int(s.shape[0]), "mean": float(s.mean()), "std": float(s.std(ddof=1)) if s.shape[0] > 1 else 0.0,
           "min": float(s.min()), "max": float(s.max())}
    for q in qs:
        out[f"p{int(q*100):02d}"] = float(s.quantile(q))
    return out

# ---------- Core calibration numbers ----------
valid_stats = qstats(df["valid_tokens_step"])
tgt_stats = qstats(df["tgt_tokens_sum"])
ctx_stats = qstats(df["ctx_tokens_sum"])
tgt_ratio_stats = qstats(df["tgt_ratio"])
tokps_stats = qstats(df["tokens_per_sec"])
samps_stats = qstats(df["samples_per_sec"])

print("valid_tokens_step stats:", valid_stats)
print("tgt_tokens_sum stats:", tgt_stats)
print("ctx_tokens_sum stats:", ctx_stats)
print("tgt_ratio stats:", tgt_ratio_stats)
print("tokens_per_sec stats:", tokps_stats)
print("samples_per_sec stats:", samps_stats)

# ---------- Token-budget accumulation sanity checks (if present) ----------
for col in ["tokens_per_update", "accum_tokens", "accum_tokens_overshoot", "accum_micro", "accum_unique_buckets"]:
    if col in df.columns:
        print(f"{col} stats:", qstats(df[col]))

# Overshoot as fraction of budget
if {"accum_tokens_overshoot", "tokens_per_update"}.issubset(df.columns):
    overshoot_frac = df["accum_tokens_overshoot"] / df["tokens_per_update"].clip(lower=1)
    print("overshoot_frac stats:", qstats(overshoot_frac))

# ---------- Bucket distribution ----------
# (C, P) frequency + token stats by bucket
bucket_cols = ["bucket_C", "bucket_P"]
if set(bucket_cols).issubset(df.columns):
    bucket_key = list(zip(df["bucket_C"].astype(int), df["bucket_P"].astype(int)))
    df["_bucket_key"] = bucket_key

    top_buckets = (
        df.groupby("_bucket_key")
          .agg(
              n_steps=("step", "count"),
              mean_valid=("valid_tokens_step", "mean"),
              p90_valid=("valid_tokens_step", lambda x: x.quantile(0.9)),
              mean_tgt=("tgt_tokens_sum", "mean"),
              mean_ctx=("ctx_tokens_sum", "mean"),
              mean_tokps=("tokens_per_sec", "mean"),
          )
          .sort_values("n_steps", ascending=False)
    )
    # print("\nTop 10 buckets by frequency:")
    # print(top_buckets.head(10))

# ---------- Recommended tokens_per_update (rule-of-thumb) ----------
# Use p90(valid_tokens_step) * k where k ~ 8..16
p90_valid = valid_stats["p90"]
for k in [8, 12, 16]:
    rec = int(np.ceil((p90_valid * k) / 256.0) * 256)  # round to 256
    # print(f"recommended tokens_per_update (p90_valid * {k}, rounded to 256): {rec}")

# ---------- Estimate steps per epoch (needs total_tokens) ----------
# Provide your total_tokens (valid tokens definition) here:
total_tokens_dataset = 9_306_127_052  # change if needed
world_size = 2  # set to 2 if running 2GPU DDP for that run

mean_valid_per_update_local = valid_stats["mean"]
mean_valid_per_update_global = mean_valid_per_update_local * world_size
steps_per_epoch = int(np.ceil(total_tokens_dataset / max(1.0, mean_valid_per_update_global)))

print(f"\nEstimated steps_per_epoch (valid tokens): {steps_per_epoch} (world_size={world_size})")

# If you want 2% / 8% equivalents:
for frac in [0.02, 0.08, 0.10, 1.0]:
    steps = int(np.ceil(steps_per_epoch * frac))
    print(f"steps for {int(frac*100)}% tokens ≈ {steps}")

# ---------- Optional: sanity check tokens_seen_total slope ----------
if "tokens_seen_total" in df.columns:
    # Rough slope: delta tokens / delta steps between first and last
    df2 = df.sort_values("step")
    dtok = df2["tokens_seen_total"].iloc[-1] - df2["tokens_seen_total"].iloc[0]
    dstep = df2["step"].iloc[-1] - df2["step"].iloc[0]
    if dstep > 0:
        print(f"\ntokens_seen_total slope ≈ {dtok / dstep:.2f} tokens/update (should be ~mean_valid)")

valid_tokens_step stats: {'count': 200, 'mean': 31302.85, 'std': 702.510485624279, 'min': 30720.0, 'max': 32680.0, 'p10': 30720.0, 'p50': 30720.0, 'p90': 32400.0, 'p95': 32400.0, 'p99': 32490.0}
tgt_tokens_sum stats: {'count': 200, 'mean': 10629.965, 'std': 1038.6924926301485, 'min': 7018.0, 'max': 14105.0, 'p10': 9319.2, 'p50': 10764.0, 'p90': 11662.2, 'p95': 12046.549999999997, 'p99': 12972.969999999996}
ctx_tokens_sum stats: {'count': 200, 'mean': 20672.885, 'std': 1115.0293782437905, 'min': 16615.0, 'max': 23702.0, 'p10': 19269.2, 'p50': 20731.0, 'p90': 22042.4, 'p95': 22282.8, 'p99': 23234.17}
tgt_ratio stats: {'count': 200, 'mean': 0.33958755153769277, 'std': 0.03271722271502027, 'min': 0.22845052083333334, 'max': 0.4591471354166667, 'p10': 0.3027701822916667, 'p50': 0.3428999083719136, 'p90': 0.374755078125, 'p95': 0.3901969401041666, 'p99': 0.42229720052083325}
tokens_per_sec stats: {'count': 200, 'mean': 3549.0220436507934, 'std': 289.52462837152814, 'min': 3072.0, 'max': 4641